# Лабораторная работа №12: открытые локальные LLM

Тема: виртуальный консультант по Python. Две реальные локальные модели:

- [Qwen2-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2-0.5B-Instruct) через Transformers/PyTorch.
- [Qwen2.5-0.5B-Instruct GGUF](https://huggingface.co/bartowski/Qwen2.5-0.5B-Instruct-GGUF) через llama-cpp-python.

Вопросы, системная инструкция и ответы на русском языке. Это разные модели:
Qwen2 и Qwen2.5, не один набор весов в разных форматах.
Обе генерируют ответы на компьютере, без RouterAI и без имитации.
Каждая модель получает инструкцию отвечать кратко и по-русски.

## Задание

##Задание на семинар
Вы планируйте разработать виртуального ассистента для консультирования по некоторой выбранной теме, используя LLM.
*   Найдите на huggingface 2 модели, которые не использовались в данном блокноте. Одна модель для работы только с помощью библиотеки transformers, другая - только llama-cpp-python.
*   Установите модели в colab. Реализуйте диалоги по отдельности с каждой из 2-х моделей с указанием системной роли и с сохранением и без сохранения истории. Продемонстрируйте некачественный ответы модели при отсутствии истории.

## Подготовка и выбор Python

В PyCharm выбери интерпретатор **Python 3.12 (.venv)**:
`.venv/bin/python` относительно папки «Пересдача».
Для Jupyter выбери ядро **Python 3**.
Пакеты должны быть установлены именно в выбранном интерпретаторе:

```bash
.venv/bin/python -m pip install -r requirements-local-models.txt
```

Выполняй ячейки сверху вниз. Первое скачивание Qwen2 занимает около 1 ГБ;
далее используются локальные файлы из кеша Hugging Face. GPU не нужен.
Ошибка `No module named torch` означает неверное окружение или отсутствие
зависимости в нём, а не ошибку модели. Путь Python напечатает первая ячейка.

In [ ]:
import os
import sys
import importlib.util

# Отключаем дополнительный загрузчик Xet и параллельность токенизатора.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Проверяем зависимости именно текущего ядра, а не системного Python.
required = ["torch", "transformers", "huggingface_hub", "llama_cpp"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
print("Текущий Python:", sys.executable)
if missing:
    raise RuntimeError(
        "Нет библиотек: " + ", ".join(missing)
        + ". Выбери Python 3.12 (.venv) или установи requirements-local-models.txt в текущее ядро."
    )

# System задаёт правила ответа, но не сообщает модели имя пользователя.
SYSTEM_ROLE = (
    "Ты извлекаешь факты из диалога. Отвечай только по-русски и строго по фактам "
    "из сообщений пользователя. Слова «я», «меня» и «мой» в сообщении пользователя "
    "относятся к пользователю, а не к ассистенту. Если пользователь сообщил имя и "
    "любимый язык, используй именно их. Если этих сведений нет в текущем диалоге, "
    "прямо скажи, что информации недостаточно."
)
QUESTIONS = [
    "Запомни два факта: имя пользователя — Михаил; любимый язык пользователя — Python. "
    "Подтверди одной короткой фразой, что факты запомнил.",
    "Как меня зовут и какой у меня любимый язык программирования?"
]

def contains_user_facts(answer):
    # Нормализуем регистр и проверяем, назвала ли модель оба факта.
    normalized = answer.lower()
    return "михаил" in normalized and "python" in normalized

def run_dialog(generate, keep_history):
    # Для каждого отдельного эксперимента начинаем с новой истории.
    history = [{"role": "system", "content": SYSTEM_ROLE}]
    transcript = []
    for question in QUESTIONS:
        # Без истории здесь всегда только system и текущий user.
        messages = history + [{"role": "user", "content": question}]
        answer = generate(messages)  # Здесь запускается выбранная локальная модель.
        transcript.append({
            "question": question, "answer": answer, "input_messages": len(messages)
        })
        print("Пользователь:", question)
        print("Модель:", answer)
        print("Передано сообщений:", len(messages), "\n")
        if keep_history:
            # Сохраняем обе реплики для следующего вызова, не изменяя веса модели.
            history.extend([
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer}
            ])
    return transcript

## 1. Transformers: загрузка Qwen2-0.5B-Instruct

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Многоязычная модель вместо английской SmolLM2: умеет отвечать по-русски.
TRANSFORMERS_MODEL = "Qwen/Qwen2-0.5B-Instruct"
torch.set_num_threads(4)  # Ограничиваем число потоков CPU на ноутбуке.

# Токенизатор переводит текст в IDs; модель загружает обученные веса.
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMERS_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    TRANSFORMERS_MODEL, dtype=torch.float32
).to("cpu")
model.eval()  # Режим инференса слоёв, не обучение.
# При жадном выборе параметры случайного выбора не применяются.
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

def transformers_generate(messages):
    # Превращаем роли и реплики в формат диалога, ожидаемый Qwen2.
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")  # PyTorch-тензоры IDs и маски.
    with torch.inference_mode():  # Не строим граф для вычисления градиентов.
        output = model.generate(
            **inputs, max_new_tokens=96, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    # Decoder-only модель возвращает вход и продолжение. Убираем входные IDs.
    generated = output[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print("Модель загружена:", TRANSFORMERS_MODEL)
print("Параметров:", sum(parameter.numel() for parameter in model.parameters()))

## 2. Transformers: без истории и с историей

In [ ]:
# В обоих режимах задаём одни и те же вопросы одной модели.
print("БЕЗ ИСТОРИИ")
tf_without = run_dialog(transformers_generate, keep_history=False)
print("С ИСТОРИЕЙ")
tf_with = run_dialog(transformers_generate, keep_history=True)

# Во втором запросе без истории фактов нет, а с историей они переданы модели.
assert tf_without[-1]["input_messages"] == 2
assert tf_with[-1]["input_messages"] == 4
assert not contains_user_facts(tf_without[-1]["answer"])
assert contains_user_facts(tf_with[-1]["answer"])


## 3. llama-cpp-python: загрузка Qwen GGUF

In [ ]:
import gc
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Освобождаем первую модель, прежде чем загружать вторую на ноутбуке с 8 ГБ RAM.
# Для повторного вызова transformers_generate после этой ячейки перезагрузи model.
del model
gc.collect()

GGUF_REPO = "bartowski/Qwen2.5-0.5B-Instruct-GGUF"
GGUF_FILE = "Qwen2.5-0.5B-Instruct-Q4_K_M.gguf"
# Скачиваем выбранный GGUF-файл или берём его из локального кеша.
model_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE)
local_llm = Llama(
    model_path=model_path,
    n_ctx=2048,        # Максимальный контекст этого экземпляра, в токенах.
    n_threads=4,      # Четыре потока CPU.
    n_gpu_layers=0,   # Не используем GPU.
    chat_format="chatml", verbose=False
)

def llama_generate(messages):
    # Вызываем локальный движок llama.cpp, а не внешний HTTP API.
    response = local_llm.create_chat_completion(
        messages=messages, max_tokens=96, temperature=0, seed=42, stream=False
    )
    # Движок возвращает словарь; извлекаем строку ответа ассистента.
    assert isinstance(response, dict), "Ожидался полный ответ, не поток"
    return response["choices"][0]["message"]["content"].strip()

print("GGUF-модель загружена:", GGUF_REPO, GGUF_FILE)

## 4. llama-cpp-python: без истории и с историей

In [ ]:
# Повторяем тот же эксперимент для другой модели и другого движка.
print("БЕЗ ИСТОРИИ")
gguf_without = run_dialog(llama_generate, keep_history=False)
print("С ИСТОРИЕЙ")
gguf_with = run_dialog(llama_generate, keep_history=True)

# Проверяем не только количество сообщений, но и смысл итогового ответа.
assert gguf_without[-1]["input_messages"] == 2
assert gguf_with[-1]["input_messages"] == 4
assert not contains_user_facts(gguf_without[-1]["answer"])
assert contains_user_facts(gguf_with[-1]["answer"])


## 5. Сравнение ответов на вопрос о предыдущей реплике

In [ ]:
# Сравниваем итоговый вопрос: во втором режиме модель видит первую реплику.
for backend, without, with_history in [
    ("Transformers / Qwen2", tf_without, tf_with),
    ("llama.cpp / Qwen2.5 GGUF", gguf_without, gguf_with)
]:
    print("\n", backend)
    print("Без истории:", without[-1]["answer"])
    print("С историей:", with_history[-1]["answer"])
    print("Факты восстановлены из истории:", contains_user_facts(with_history[-1]["answer"]))


## Вывод

Обе модели выполняют генерацию локально и получают одинаковую русскую
системную инструкцию и вопросы. Во втором запросе без истории передаются только
системная роль и текущий вопрос, поэтому имени Михаил в контексте нет. Модель
может признать нехватку данных или выдумать ответ — это и есть некачественный
результат при отсутствии истории.

В режиме с историей во втором вызове передаются четыре сообщения: системная
роль, первая реплика пользователя, первый ответ ассистента и новый вопрос.
Поэтому модель может извлечь из контекста два факта: имя Михаил и язык Python.
Это изменение входного контекста, а не обучение или изменение весов модели.
Проверки `assert` подтверждают не только количество сообщений, но и наличие
обоих фактов в ответе с историей.
